In [8]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import warnings

In [9]:
# 1. Load your cleaned data from the EDA phase
# Adjust the path if your data folder is named differently
df = pd.read_parquet('../data/network_intrusion_cleaned.parquet')

print(f"Original Dataset Shape: {df.shape}")
print("\nClass Distribution Before Sampling:")
print(df['Label'].value_counts())

Original Dataset Shape: (2517883, 26)

Class Distribution Before Sampling:
Label
BENIGN                        2092335
DoS Hulk                       172688
DDoS                           127995
PortScan                        90694
DoS GoldenEye                   10281
FTP-Patator                      5927
DoS slowloris                    5385
DoS Slowhttptest                 5228
SSH-Patator                      3217
Bot                              1948
Web Attack � Brute Force         1470
Web Attack � XSS                  652
Infiltration                       35
Web Attack � Sql Injection         21
Heartbleed                          7
Name: count, dtype: int64


In [10]:
# Define threshold for "rare" classes and your sample fraction
THRESHOLD = 50 
SAMPLE_FRACTION = 0.20

# Count instances per class
class_counts = df['Label'].value_counts()

# Separate classes into rare and abundant
rare_classes = class_counts[class_counts <= THRESHOLD].index
abundant_classes = class_counts[class_counts > THRESHOLD].index

# 1. Keep 100% of the rare classes to preserve critical minority attacks
df_rare = df[df['Label'].isin(rare_classes)]

# 2. Perform 20% stratified sampling on the abundant classes
df_abundant = df[df['Label'].isin(abundant_classes)]
_, df_abundant_sampled = train_test_split(
    df_abundant, 
    test_size=SAMPLE_FRACTION, 
    random_state=42, 
    stratify=df_abundant['Label']
)

# 3. Combine both sets back together and shuffle them
df_sampled = pd.concat([df_rare, df_abundant_sampled])
df_sampled = df_sampled.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Sampled Dataset Shape: {df_sampled.shape}")
print("\nClass Distribution After Smart Sampling:")
print(df_sampled['Label'].value_counts())

Sampled Dataset Shape: (503627, 26)

Class Distribution After Smart Sampling:
Label
BENIGN                        418467
DoS Hulk                       34538
DDoS                           25599
PortScan                       18139
DoS GoldenEye                   2056
FTP-Patator                     1185
DoS slowloris                   1077
DoS Slowhttptest                1046
SSH-Patator                      643
Bot                              390
Web Attack � Brute Force         294
Web Attack � XSS                 130
Infiltration                      35
Web Attack � Sql Injection        21
Heartbleed                         7
Name: count, dtype: int64


In [11]:
import json
from sklearn.preprocessing import LabelEncoder

# Initialize LabelEncoder
le = LabelEncoder()

# Fit and transform the Label column
df_sampled['Label_Encoded'] = le.fit_transform(df_sampled['Label'])

# Create a mapping dictionary (converting numpy types to native Python types for JSON)
label_mapping = {str(name): int(num) for name, num in zip(le.classes_, le.transform(le.classes_))}

print("Target Label Mapping for Classification:")
for name, num in label_mapping.items():
    print(f"{num}: {name}")

# Save the mapping to a JSON file
with open('../data/label_mapping.json', 'w') as f:
    json.dump(label_mapping, f, indent=4)
print("\nLabel mapping successfully saved to 'data/label_mapping.json'")

# Drop the old string column and rename the new one back to 'Label'
df_sampled = df_sampled.drop(columns=['Label'])
df_sampled = df_sampled.rename(columns={'Label_Encoded': 'Label'})

Target Label Mapping for Classification:
0: BENIGN
1: Bot
2: DDoS
3: DoS GoldenEye
4: DoS Hulk
5: DoS Slowhttptest
6: DoS slowloris
7: FTP-Patator
8: Heartbleed
9: Infiltration
10: PortScan
11: SSH-Patator
12: Web Attack � Brute Force
13: Web Attack � Sql Injection
14: Web Attack � XSS

Label mapping successfully saved to 'data/label_mapping.json'


In [12]:
# Export the final sampled and encoded dataset
df_sampled.to_parquet('../data/network_intrusion_sampled.parquet', index=False)
print("Successfully exported to 'data/network_intrusion_sampled.parquet'.")
print("Ready for Classification!")

Successfully exported to 'data/network_intrusion_sampled.parquet'.
Ready for Classification!


In [13]:
import json

# 1. Load the mapping
with open('../data/label_mapping.json', 'r') as f:
    label_mapping = json.load(f)

# 2. Reverse the dictionary so numbers are the keys (e.g., 0: 'BENIGN')
inverse_mapping = {v: k for k, v in label_mapping.items()}

# 3. Example: Using it to label a classification report
# target_names takes a list of strings ordered by their class index (0, 1, 2...)
target_names = [inverse_mapping[i] for i in range(len(inverse_mapping))]

# (Assume y_test and y_pred are your model's outputs)
# print(classification_report(y_test, y_pred, target_names=target_names))